# Semantic Search & LLM Application
**Starter Notebook: Assignment Sesi 13**

Notebook ini adalah implementasi prototype aplikasi **semantic search** yang terintegrasi dengan **LLM**, menggunakan knowledge base FAQ e-commerce.

Alur besar: **Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi**.

> **Catatan dual-provider:** notebook ini mendukung dua jalur API.
> - Jika `GEMINI_API_KEY` tersedia dan region mendukung → pakai **Gemini API langsung** (`gemini-embedding-001`, `gemini-3.6-flash`).
> - Jika tidak (misal terkena pembatasan region) → otomatis **fallback ke OpenRouter** (model yang sama: `google/gemini-embedding-001`, `google/gemini-3.6-flash`).
>
> Satu jalur dipakai konsisten dari awal sampai akhir, sehingga vektor embedding tidak tercampur antar model.

## 1. Import Library & Load Knowledge Base

In [1]:
import os
import json
import time
import urllib.request
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Baca API key dari file .env (jika ada) atau environment variable
load_dotenv(Path.cwd() / ".env", override=False)
load_dotenv(Path.cwd().parent / ".env", override=False)

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')

print('GEMINI_API_KEY tersedia:', bool(GEMINI_API_KEY))
print('OPENROUTER_API_KEY tersedia:', bool(OPENROUTER_API_KEY))

# Load knowledge base
df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

GEMINI_API_KEY tersedia: True
OPENROUTER_API_KEY tersedia: True
Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai **satu unit** yang akan di-embed dan disimpan ke vector database. Pertanyaan dan jawaban digabung supaya konteksnya utuh saat di-retrieve.

In [2]:
# Gabungkan question dan answer menjadi satu teks representatif
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()

print('Contoh unit retrievable:')
print(documents[0])
print()
print(f'Total dokumen: {len(documents)}')

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.

Total dokumen: 59


## 3. Membuat Embeddings

Setiap unit retrievable diubah menjadi **vektor embedding** menggunakan **Gemini Embeddings API** (`gemini-embedding-001`, dimensi 768).

Karena region server ini tidak mendukung akses langsung ke Gemini API, notebook otomatis memilih jalur:
- **Jalur A (default):** Gemini API langsung: `gemini-embedding-001` dengan `task_type` (`RETRIEVAL_DOCUMENT` untuk dokumen, `RETRIEVAL_QUERY` untuk query).
- **Jalur B (fallback):** OpenRouter: `google/gemini-embedding-001` (dimensi 3072, tanpa `task_type`).

> **Penting:** satu jalur dipakai konsisten untuk seluruh dokumen DAN query, supaya ruang vektornya seragam. Normalisasi L2 diterapkan agar inner product = cosine similarity.

In [3]:
# ===== Konfigurasi model embedding =====
EMBED_MODEL = 'gemini-embedding-001'
EMBED_DIM = 768  # dimensi via Gemini langsung (OpenRouter: 3072)
BATCH_SIZE = 20

# Deteksi jalur: coba Gemini langsung dulu
USE_GEMINI = False
if GEMINI_API_KEY:
    try:
        _client = genai.Client(api_key=GEMINI_API_KEY)
        _client.models.embed_content(
            model=EMBED_MODEL,
            contents=['uji koneksi'],
            config=types.EmbedContentConfig(
                task_type='RETRIEVAL_DOCUMENT',
                output_dimensionality=EMBED_DIM,
            ),
        )
        USE_GEMINI = True
        print('Jalur embedding: Gemini API langsung (dimensi 768)')
    except Exception as e:
        print('Gemini langsung gagal, fallback ke OpenRouter:', str(e)[:100])

if not USE_GEMINI:
    assert OPENROUTER_API_KEY, 'Tidak ada API key tersedia (GEMINI_API_KEY / OPENROUTER_API_KEY)'
    print('Jalur embedding: OpenRouter (dimensi 3072)')


def embed_texts(texts):
    """Embedding daftar teks -> np.float32 (n, dim), sudah dinormalisasi L2."""
    if isinstance(texts, str):
        texts = [texts]
    texts = list(texts)

    if USE_GEMINI:
        # Jalur A: Gemini API langsung
        vecs = []
        for start in range(0, len(texts), BATCH_SIZE):
            chunk = texts[start:start + BATCH_SIZE]
            result = _client.models.embed_content(
                model=EMBED_MODEL,
                contents=chunk,
                config=types.EmbedContentConfig(
                    task_type='RETRIEVAL_DOCUMENT',
                    output_dimensionality=EMBED_DIM,
                ),
            )
            vecs.extend(e.values for e in result.embeddings)
        v = np.array(vecs, dtype='float32')
    else:
        # Jalur B: OpenRouter embeddings
        payload = {'model': 'google/gemini-embedding-001', 'input': texts}
        req = urllib.request.Request(
            'https://openrouter.ai/api/v1/embeddings',
            data=json.dumps(payload).encode(),
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}',
                     'Content-Type': 'application/json'},
        )
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        data = sorted(d['data'], key=lambda x: x['index'])
        v = np.array([item['embedding'] for item in data], dtype='float32')

    # Normalisasi L2 (agar inner product = cosine similarity)
    norms = np.linalg.norm(v, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    return v / norms


# Embed seluruh dokumen knowledge base
print(f'Embedding {len(documents)} dokumen...')
t0 = time.time()
embeddings = embed_texts(documents)
print(f'Shape embeddings: {embeddings.shape} ({time.time()-t0:.1f}s)')

Gemini langsung gagal, fallback ke OpenRouter: 400 FAILED_PRECONDITION. {'error': {'code': 400, 'message': 'User location is not supported for the 
Jalur embedding: OpenRouter (dimensi 3072)
Embedding 59 dokumen...


Shape embeddings: (59, 3072) (4.8s)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

Embedding disimpan ke **FAISS** menggunakan `IndexFlatIP` (inner product). Karena vektor sudah dinormalisasi L2, inner product = **cosine similarity**: cocok untuk semantic search.

In [4]:
# Bangun FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print('Dimensi index:', index.d)
print('Jumlah vektor tersimpan di index:', index.ntotal)
assert index.ntotal == len(documents), 'Jumlah vektor tidak sesuai jumlah entri KB!'

Dimensi index: 3072
Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi `semantic_search(query, top_k)`:
1. Mengubah query menjadi embedding (dengan jalur yang sama seperti dokumen).
2. Mencari `top_k` vektor terdekat di index FAISS.
3. Mengembalikan entri FAQ paling relevan beserta similarity score.

In [5]:
def embed_query(query):
    """Embedding query (satu teks). Konsisten dengan jalur dokumen."""
    if USE_GEMINI:
        result = _client.models.embed_content(
            model=EMBED_MODEL,
            contents=[query],
            config=types.EmbedContentConfig(
                task_type='RETRIEVAL_QUERY',  # task_type khusus query
                output_dimensionality=EMBED_DIM,
            ),
        )
        v = np.array([e.values for e in result.embeddings], dtype='float32')
    else:
        payload = {'model': 'google/gemini-embedding-001', 'input': [query]}
        req = urllib.request.Request(
            'https://openrouter.ai/api/v1/embeddings',
            data=json.dumps(payload).encode(),
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}',
                     'Content-Type': 'application/json'},
        )
        d = json.loads(urllib.request.urlopen(req, timeout=60).read())
        v = np.array([d['data'][0]['embedding']], dtype='float32')
    norms = np.linalg.norm(v, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    return v / norms


def semantic_search(query, top_k=3):
    query_embedding = embed_query(query)
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        if idx == -1:
            continue
        results.append({
            'rank': rank,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'similarity': float(score),
        })
    return results


# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(f"#{r['rank']} [{r['category']}] sim={r['similarity']:.4f}")
    print(f"   Q: {r['question']}")
    print(f"   A: {r['answer'][:80]}...")

#1 [Pengiriman] sim=0.8332
   Q: Bagaimana jika paket saya belum sampai melebihi estimasi?
   A: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi custo...
#2 [Pengiriman] sim=0.7956
   Q: Kenapa status pengiriman saya tidak berubah selama beberapa hari?
   A: Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Dis...
#3 [Pengiriman] sim=0.7840
   Q: Kenapa nomor resi saya belum bisa dilacak di website kurir?
   A: Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, bia...


## 6. Mendesain Prompt dan Context

Prompt disusun secara **terstruktur** dengan empat komponen:
- **Persona:** asisten customer service e-commerce yang ramah.
- **Task:** menjawab pertanyaan HANYA berdasarkan FAQ.
- **Context:** hasil semantic search (top-K FAQ relevan).
- **Format:** jawaban natural dalam Bahasa Indonesia.

Prompt juga mengarahkan LLM untuk **mengakui ketidaktahuan** jika informasi tidak tersedia di knowledge base: ini penting supaya model tidak mengarang jawaban.

In [6]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi dari FAQ berikut.
Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut
dan sarankan pengguna menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

Fungsi `generate_answer(query)` menggabungkan alur lengkap:
**query → retrieval (semantic search) → prompt (dengan context) → LLM → jawaban**.

Model LLM:
- Jalur Gemini langsung: `gemini-3.6-flash` (versi terbaru; `gemini-2.0-flash` di starter sudah tidak tersedia).
- Fallback OpenRouter: `google/gemini-3.6-flash` (model yang sama).

> Catatan: `max_tokens` dibuat cukup besar (500) karena Gemini 3.x memakai sebagian jatah untuk reasoning internal sebelum menjawab.

In [7]:
LLM_MODEL = 'gemini-3.6-flash'  # via Gemini langsung
LLM_MODEL_OR = 'google/gemini-3.6-flash'  # via OpenRouter


def generate_answer(query, top_k=3, max_tokens=500):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    if USE_GEMINI:
        # Jalur A: Gemini API langsung
        response = _client.models.generate_content(
            model=LLM_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.3, max_output_tokens=max_tokens),
        )
        answer = response.text
    else:
        # Jalur B: OpenRouter chat completions
        payload = {
            'model': LLM_MODEL_OR,
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0.3,
            'max_tokens': max_tokens,
        }
        req = urllib.request.Request(
            'https://openrouter.ai/api/v1/chat/completions',
            data=json.dumps(payload).encode(),
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}',
                     'Content-Type': 'application/json'},
        )
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        answer = d['choices'][0]['message'].get('content') or ''

    return answer, search_results


# Contoh percobaan
answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print(f"  - [{s['category']}] {s['question']} (sim={s['similarity']:.4f})")

Jawaban LLM:
Halo! Ya, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. 

Ada hal lain yang bisa saya bantu?

Sumber FAQ yang digunakan:
  - [Pembayaran] Apakah saya bisa membayar dengan cicilan? (sim=0.8954)
  - [Pembayaran] Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (sim=0.7257)
  - [Pembayaran] Metode pembayaran apa saja yang tersedia? (sim=0.6947)


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Sistem diuji dengan **6 query** yang mencakup tiga kategori sesuai instruksi:
- **2 query yang jelas ada jawabannya di FAQ** (kata persis).
- **2 query dengan kata/frasa berbeda dari FAQ asli** (parafrase/sinonim): untuk menguji apakah semantic search menangkap *makna*, bukan sekadar mencocokkan kata kunci.
- **2 query yang TIDAK ada jawabannya di FAQ**: untuk menguji bagaimana sistem merespons ketidaktahuannya.

In [8]:
test_queries = [
    # (label, query)
    ('langsung', 'Bagaimana cara melakukan pemesanan di platform ini?'),
    ('langsung', 'Berapa lama estimasi waktu pengiriman barang?'),
    ('parafrase', 'Gimana caranya saya order barang di toko kalian?'),
    ('parafrase', 'Paketku kok belum datang ya, kapan nyampe?'),
    ('tidak-ada', 'Bagaimana cara klaim asuransi kesehatan?'),
    ('tidak-ada', 'Apakah kalian menjual tiket pesawat?'),
]

all_results = []
for i, (label, q) in enumerate(test_queries, 1):
    print(f'\n{"="*60}')
    print(f'QUERY {i} [{label}]')
    print(f'Q: {q}')
    print('-' * 60)

    answer, sources = generate_answer(q)

    print('Retrieval (top-3):')
    for s in sources:
        print(f"  #{s['rank']} [{s['category']}] sim={s['similarity']:.4f} | {s['question'][:60]}")
    print()
    print('Jawaban LLM:')
    print(answer[:400])

    all_results.append({
        'query': q,
        'label': label,
        'sources': sources,
        'answer': answer,
    })
    time.sleep(0.5)  # jeda singkat antar panggilan


QUERY 1 [langsung]
Q: Bagaimana cara melakukan pemesanan di platform ini?
------------------------------------------------------------


Retrieval (top-3):
  #1 [Pemesanan] sim=0.8578 | Bagaimana cara melakukan pemesanan di platform ini?
  #2 [Pemesanan] sim=0.7482 | Apakah saya bisa memesan tanpa membuat akun?
  #3 [Pemesanan] sim=0.7148 | Bagaimana cara mengetahui status pesanan saya saat ini?

Jawaban LLM:
Halo! Untuk melakukan pemesanan di platform ini, Anda cukup memilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke



QUERY 2 [langsung]
Q: Berapa lama estimasi waktu pengiriman barang?
------------------------------------------------------------


Retrieval (top-3):
  #1 [Pengiriman] sim=0.9026 | Berapa lama estimasi waktu pengiriman barang?
  #2 [Pengiriman] sim=0.7833 | Bagaimana jika paket saya belum sampai melebihi estimasi?
  #3 [Pemesanan] sim=0.7559 | Berapa lama waktu yang dibutuhkan sebelum pesanan diproses p

Jawaban LLM:
Halo! Estimasi pengiriman barang umumnya adalah 2-5 hari kerja untuk area dalam kota dan



QUERY 3 [parafrase]
Q: Gimana caranya saya order barang di toko kalian?
------------------------------------------------------------


...

Retrieval (top-3):
  #1 [Pengiriman] sim=0.8103 | Bagaimana jika paket saya belum sampai melebihi estimasi?
  #2 [Pengiriman] sim=0.7921 | Berapa lama estimasi waktu pengiriman barang?
  #3 [Pengiriman] sim=0.7676 | Bagaimana cara melacak paket yang sedang dikirim?

Jawaban LLM:
Halo Kak! 

Estimasi waktu pengiriman barang secara umum adalah **2-5 hari kerja



QUERY 5 [tidak-ada]
Q: Bagaimana cara klaim asuransi kesehatan?
------------------------------------------------------------


Retrieval (top-3):
  #1 [Pengiriman] sim=0.6467 | Apakah ongkos kirim sudah termasuk asuransi pengiriman?
  #2 [Pengiriman] sim=0.6351 | Apa yang harus dilakukan jika paket diterima dalam kondisi r
  #3 [Pemesanan] sim=0.6311 | Bagaimana cara melakukan pemesanan di platform ini?

Jawaban LLM:
Halo! Mohon maaf, saya belum memiliki informasi mengenai cara klaim asuransi kesehatan. 

Untuk bantuan lebih lanjut terkait pertanyaan Anda, silakan hubungi tim customer service kami secara langsung, ya. Terima kasih!



QUERY 6 [tidak-ada]
Q: Apakah kalian menjual tiket pesawat?
------------------------------------------------------------


Retrieval (top-3):
  #1 [Pembayaran] sim=0.6499 | Metode pembayaran apa saja yang tersedia?
  #2 [Pemesanan] sim=0.6356 | Apakah saya bisa memesan tanpa membuat akun?
  #3 [Pemesanan] sim=0.6250 | Apakah bisa memesan dalam jumlah besar untuk kebutuhan resel

Jawaban LLM:
Halo! Mohon maaf, saat ini saya belum memiliki informasi mengenai penjualan tiket pesawat. Untuk pertanyaan lebih lanjut, Anda bisa langsung menghubungi customer service kami ya. Terima kasih!


### Catatan Analisis Relevansi

Pengamatan terhadap hasil pengujian di atas:

**Query langsung (kata persis):** retrieval sangat akurat. Top-1 selalu entri FAQ yang tepat dengan similarity tinggi (0.85+). LLM menjawab dengan baik karena context-nya relevan.

**Query parafrase (kata beda):** semantic search tetap menemukan FAQ yang tepat karena menangkap *makna*, bukan kata. Contoh: *"Gimana caranya saya order barang di toko kalian?"* → ter-retrieve *"Bagaimana cara melakukan pemesanan di platform ini?"* dengan similarity 0.84. Ini membuktikan embedding menangkap kesamaan semantik.

**Query di luar KB:** similarity semua hasil rendah (< 0.65). Ini sinyal penting: LLM harus dilatih (lewat prompt) untuk **mengakui ketidaktahuan** ketika skor retrieval rendah. Pada pengujian, LLM merespons dengan baik: menyebut bahwa info tidak tersedia dan menyarankan hubungi customer service.

**Keterbatasan:** FAISS `IndexFlatIP` melakukan pencarian exact (brute force): cocok untuk 59 entri, tapi kurang skalabel untuk jutaan dokumen (bisa diganti `IndexIVFFlat`/`HNSW`).